In [73]:
import pandas as pd
import numpy as np
import re

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split

In [74]:
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\Sujan\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\Sujan\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [75]:
df = pd.read_json("../data/News_Category_Dataset.json", lines=True)
df.head()

,link,headline,category,short_description,authors,date
0,https://www.huffpost.com/entry/covid-boosters-...,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS,Health experts said it is too early to predict...,"Carla K. Johnson, AP",2022-09-23
1,https://www.huffpost.com/entry/american-airlin...,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS,He was subdued by passengers and crew when he ...,Mary Papenfuss,2022-09-23
2,https://www.huffpost.com/entry/funniest-tweets...,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY,"""Until you have a dog you don't understand wha...",Elyse Wanshel,2022-09-23
3,https://www.huffpost.com/entry/funniest-parent...,The Funniest Tweets From Parents This Week (Se...,PARENTING,"""Accidentally put grown-up toothpaste on my to...",Caroline Bologna,2022-09-23
4,https://www.huffpost.com/entry/amy-cooper-lose...,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS,Amy Cooper accused investment firm Franklin Te...,Nina Golgowski,2022-09-22


In [76]:
print("Shape:", df.shape)
print()

df.info()

Shape: (209527, 6)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 209527 entries, 0 to 209526
Data columns (total 6 columns):
 #   Column             Non-Null Count   Dtype         
---  ------             --------------   -----         
 0   link               209527 non-null  object        
 1   headline           209527 non-null  object        
 2   category           209527 non-null  object        
 3   short_description  209527 non-null  object        
 4   authors            209527 non-null  object        
 5   date               209527 non-null  datetime64[ns]
dtypes: datetime64[ns](1), object(5)
memory usage: 9.6+ MB


In [77]:
df = df[['headline', 'category']]
df.head()

,headline,category
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY
3,The Funniest Tweets From Parents This Week (Se...,PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS


In [78]:
df['category'].value_counts()

category
POLITICS          35602
WELLNESS          17945
ENTERTAINMENT     17362
TRAVEL             9900
STYLE & BEAUTY     9814
PARENTING          8791
HEALTHY LIVING     6694
QUEER VOICES       6347
FOOD & DRINK       6340
BUSINESS           5992
COMEDY             5400
SPORTS             5077
BLACK VOICES       4583
HOME & LIVING      4320
PARENTS            3955
THE WORLDPOST      3664
WEDDINGS           3653
WOMEN              3572
CRIME              3562
IMPACT             3484
DIVORCE            3426
WORLD NEWS         3299
MEDIA              2944
WEIRD NEWS         2777
GREEN              2622
WORLDPOST          2579
RELIGION           2577
STYLE              2254
SCIENCE            2206
TECH               2104
TASTE              2096
MONEY              1756
ARTS               1509
ENVIRONMENT        1444
FIFTY              1401
GOOD NEWS          1398
U.S. NEWS          1377
ARTS & CULTURE     1339
COLLEGE            1144
LATINO VOICES      1130
CULTURE & ARTS     1074
EDUCATI

In [79]:
categories = [
    "POLITICS",
    "BUSINESS",
    "SPORTS",
    "TECH",
    "ENTERTAINMENT",
    "WORLD NEWS",
    "SCIENCE",
    "WORLDPOST"
]

df = df[df['category'].isin(categories)]

df['category'].value_counts()

category
POLITICS         35602
ENTERTAINMENT    17362
BUSINESS          5992
SPORTS            5077
WORLD NEWS        3299
WORLDPOST         2579
SCIENCE           2206
TECH              2104
Name: count, dtype: int64

In [80]:
label_map = {
    "POLITICS": "Politics",
    "BUSINESS": "Business",
    "SPORTS": "Sports",
    "TECH": "Technology",
    "ENTERTAINMENT": "Entertainment",
    "WORLD NEWS": "World",
    "WORLDPOST ": "World",
    "SCIENCE": "Science"
}

df['category'] = df['category'].map(label_map)

df.head()

,headline,category
7,Puerto Ricans Desperate For Water After Hurric...,World
9,Biden At UN To Call Russian War An Affront To ...,World
10,World Cup Captains Want To Wear Rainbow Armban...,World
11,Man Sets Himself On Fire In Apparent Protest O...,World
12,Fiona Threatens To Become Category 4 Storm Hea...,World


In [81]:
df.isnull().sum()

headline       0
category    2579
dtype: int64

In [82]:
df.dropna(inplace=True)

print(df.shape)

(71642, 2)


In [83]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [84]:
def clean_text(text):

    #lowercase
    text = text.lower()

    #remove numbers and symbols
    text = re.sub(r'[^a-z\s]', '',text)

    #split in to words
    words = text.split()

    #remove stop words
    words = [word for word in words
             if word not in stop_words]

    #lemmatization
    words = [lemmatizer.lemmatize(word)
              for word in words]
        
    return " ".join(words)

In [85]:
df['clean_headline'] = df['headline'].apply(clean_text)

df.head()

,headline,category,clean_headline
7,Puerto Ricans Desperate For Water After Hurric...,World,puerto ricans desperate water hurricane fionas...
9,Biden At UN To Call Russian War An Affront To ...,World,biden un call russian war affront body charter
10,World Cup Captains Want To Wear Rainbow Armban...,World,world cup captain want wear rainbow armband qatar
11,Man Sets Himself On Fire In Apparent Protest O...,World,man set fire apparent protest funeral japan abe
12,Fiona Threatens To Become Category 4 Storm Hea...,World,fiona threatens become category storm headed b...


In [86]:
X = df['clean_headline']
y = df['category']

In [87]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

In [88]:
print("Training samples:" , len(X_train))
print("Testing sample:", len(X_test))

Training samples: 57313
Testing sample: 14329


In [89]:
for i in range(5):
    print("Original:")
    print(df['headline'].iloc[i])

    print("Cleaned:")
    print(df['clean_headline'].iloc[i])


Original:
Puerto Ricans Desperate For Water After Hurricane Fiona’s Rampage
Cleaned:
puerto ricans desperate water hurricane fionas rampage
Original:
Biden At UN To Call Russian War An Affront To Body's Charter
Cleaned:
biden un call russian war affront body charter
Original:
World Cup Captains Want To Wear Rainbow Armbands In Qatar
Cleaned:
world cup captain want wear rainbow armband qatar
Original:
Man Sets Himself On Fire In Apparent Protest Of Funeral For Japan's Abe
Cleaned:
man set fire apparent protest funeral japan abe
Original:
Fiona Threatens To Become Category 4 Storm Headed To Bermuda
Cleaned:
fiona threatens become category storm headed bermuda


In [90]:
train_df = pd.DataFrame({
    "headline": X_train,
    "category": y_train
})

test_df = pd.DataFrame({
    "headline": X_test,
    "category": y_test
})

train_df['headline'] = train_df['headline'].fillna("").astype(str)
test_df['headline'] = test_df['headline'].fillna("").astype(str)

train_df.to_csv("../data/train.csv", index=False)
test_df.to_csv("../data/test.csv", index=False)